In [1]:
import json
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set up Hugging Face cache directory
cache_dir = Path(r"C:/Users/User/Documents/devanasokan_fyp/huggingface_cache")
cache_dir.mkdir(parents=True, exist_ok=True)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", cache_dir=str(cache_dir))
print(f"Tokenizer vocab size: {len(tokenizer)}")

c:\Users\User\Documents\devanasokan_fyp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Tokenizer vocab size: 30522


# LSTM Lyrics Classifier

This notebook builds a full text classification workflow for lyrics using an LSTM model:

1. Load the lyrics dataset.
2. Clean and normalize the text.
3. Tokenize and build a vocabulary from the training split.
4. Convert lyrics into padded integer sequences.
5. Train and evaluate an LSTM classifier in PyTorch.

In [2]:
data_path = Path(r"C:/Users/User/Documents/devanasokan_fyp/preparation/modeldata.csv")
df = pd.read_csv(data_path)
df = df[["verse", "label"]].dropna().drop_duplicates().reset_index(drop=True)

print(df.shape)
print(df["label"].value_counts().sort_index())

(22878, 2)
label
0    11439
1    11439
Name: count, dtype: int64


#### Text Cleaning and Train/Validation/Test Split

The LSTM should only see text from the training split when building the vocabulary. That avoids leaking information from validation or test lyrics into the tokenizer.


In [3]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text) # Replace non-alphanumeric characters with spaces
    text = re.sub(r"\s+", " ", text).strip() # Replace multiple spaces with a single space and trim
    return text

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=seed,
    stratify=df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=seed,
    stratify=temp_df["label"],
)

for frame in (train_df, val_df, test_df):
    frame.loc[:, "clean_verse"] = frame["verse"].apply(clean_text)

print("train:", train_df.shape, train_df["label"].value_counts().sort_index().to_dict())
print("val:", val_df.shape, val_df["label"].value_counts().sort_index().to_dict())
print("test:", test_df.shape, test_df["label"].value_counts().sort_index().to_dict())

train: (18302, 3) {0: 9151, 1: 9151}
val: (2288, 3) {0: 1144, 1: 1144}
test: (2288, 3) {0: 1144, 1: 1144}


In [4]:
MAX_LEN = 180


def tokenize(text: str) -> list[str]:
    return tokenizer.tokenize(text)


counter = Counter()
for text in train_df["clean_verse"]:
    counter.update(tokenize(text))

vocab = tokenizer.get_vocab()

print(f"Vocabulary size: {len(vocab)}")
print("Most common tokens:", counter.most_common(10))

Token indices sequence length is longer than the specified maximum sequence length for this model (544 > 512). Running this sequence through the model will result in indexing errors


Vocabulary size: 30522
Most common tokens: [('i', 54906), ('you', 40629), ('the', 31056), ('to', 24273), ('it', 20227), ('a', 19852), ('me', 18337), ('and', 17799), ('not', 15771), ('is', 15142)]


In [5]:
def numericalize(text: str) -> tuple[torch.Tensor, torch.Tensor]:
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )
    if not token_ids:
        token_ids = [tokenizer.unk_token_id]
    length = len(token_ids)
    if len(token_ids) < MAX_LEN:
        token_ids += [tokenizer.pad_token_id] * (MAX_LEN - len(token_ids))
    return torch.tensor(token_ids, dtype=torch.long), torch.tensor(length, dtype=torch.long)

sample_ids, sample_length = numericalize(train_df.iloc[0]["clean_verse"] )
print("sample length:", sample_length.item())
print("sample ids:", sample_ids[:20].tolist())

sample length: 46
sample ids: [2026, 2026, 2092, 2009, 2003, 25085, 2066, 2057, 2024, 2035, 2183, 2000, 3280, 2035, 2183, 2000, 3280, 9061, 9061, 2524]


In [6]:
class LyricsDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.texts = frame["clean_verse"].tolist()
        self.labels = frame["label"].astype(np.float32).tolist()

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int):
        input_ids, length = numericalize(self.texts[index])
        label = torch.tensor(self.labels[index], dtype=torch.float32)
        return input_ids, length, label


batch_size = 64
train_loader = DataLoader(LyricsDataset(train_df), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(LyricsDataset(val_df), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(LyricsDataset(test_df), batch_size=batch_size, shuffle=False)

batch_input_ids, batch_lengths, batch_labels = next(iter(train_loader))
print(batch_input_ids.shape, batch_lengths.shape, batch_labels.shape)

torch.Size([64, 180]) torch.Size([64]) torch.Size([64])


LSTM Model Definition

The model uses an embedding layer, an LSTM encoder, dropout, and a linear output layer for binary classification.


In [7]:
class LSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 128,
        hidden_dim: int = 128,
        num_layers: int = 2,
        bidirectional: bool = True,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=tokenizer.pad_token_id)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(output_dim, 1)

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, (hidden, _) = self.lstm(packed)
        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]
        logits = self.fc(self.dropout(hidden))
        return logits.squeeze(1)


model = LSTMClassifier(vocab_size=len(tokenizer)).to(device)
print(model)

LSTMClassifier(
  (embedding): Embedding(30522, 128, padding_idx=0)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)


In [8]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def run_epoch(loader: DataLoader, training: bool = True):
    model.train() if training else model.eval()

    total_loss = 0.0
    all_predictions = []
    all_targets = []

    for input_ids, lengths, labels in loader:
        input_ids = input_ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(training):
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        predictions = (torch.sigmoid(logits) >= 0.5).long()
        all_predictions.extend(predictions.detach().cpu().tolist())
        all_targets.extend(labels.detach().cpu().long().tolist())
        total_loss += loss.item() * input_ids.size(0)

    average_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_targets, all_predictions)
    return average_loss, accuracy, all_targets, all_predictions


with torch.no_grad():
    preview_logits = model(batch_input_ids.to(device), batch_lengths.to(device))
print("preview logits shape:", preview_logits.shape)

preview logits shape: torch.Size([64])


In [9]:
num_epochs = 5
best_val_loss = float("inf")
best_model_path = Path("lstm_lyrics_classifier.pt")
history = []

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc, _, _ = run_epoch(train_loader, training=True)
    val_loss, val_acc, _, _ = run_epoch(val_loader, training=False)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)

history_df = pd.DataFrame(history)
history_df

Epoch 01 | train_loss=0.6423 train_acc=0.6298 | val_loss=0.5960 val_acc=0.6971
Epoch 02 | train_loss=0.5974 train_acc=0.6816 | val_loss=0.5792 val_acc=0.7037
Epoch 03 | train_loss=0.5320 train_acc=0.7337 | val_loss=0.5392 val_acc=0.7233
Epoch 04 | train_loss=0.4453 train_acc=0.7938 | val_loss=0.4849 val_acc=0.7675
Epoch 05 | train_loss=0.3888 train_acc=0.8279 | val_loss=0.4530 val_acc=0.7968


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,0.642346,0.629767,0.595953,0.697115
1,2,0.597419,0.681565,0.579178,0.703671
2,3,0.532047,0.733745,0.539230,0.723339
3,4,0.445265,0.793793,0.484898,0.767483
4,5,0.388809,0.827942,0.453005,0.796766


Final Evaluation and Saving

After training, reload the best checkpoint and evaluate it once on the test split. Then save the model and vocabulary together for inference.

In [10]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
test_loss, test_acc, test_targets, test_predictions = run_epoch(test_loader, training=False)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print(classification_report(test_targets, test_predictions, digits=4))

Test loss: 0.4531
Test accuracy: 0.7858
              precision    recall  f1-score   support

           0     0.7858    0.7858    0.7858      1144
           1     0.7858    0.7858    0.7858      1144

    accuracy                         0.7858      2288
   macro avg     0.7858    0.7858    0.7858      2288
weighted avg     0.7858    0.7858    0.7858      2288



In [11]:
artifact_dir = Path("lstm_artifacts")
artifact_dir.mkdir(exist_ok=True)

model_path = artifact_dir / "lyrics_lstm.pt"
vocab_path = artifact_dir / "lyrics_vocab.json"

torch.save(model.state_dict(), model_path)
with open(vocab_path, "w", encoding="utf-8") as vocab_file:
    json.dump(vocab, vocab_file, ensure_ascii=False, indent=2)


def predict_text(text: str):
    model.eval()
    cleaned_text = clean_text(text)
    input_ids, length = numericalize(cleaned_text)
    with torch.no_grad():
        logits = model(input_ids.unsqueeze(0).to(device), length.unsqueeze(0).to(device))
        probability = torch.sigmoid(logits).item()
        prediction = int(probability >= 0.5)
    return probability, prediction


sample_probability, sample_prediction = predict_text(" you in the dark")
print({"probability": sample_probability, "prediction": sample_prediction})

{'probability': 0.27243149280548096, 'prediction': 0}
